# 10 — Descriptive replication of Lorigan et al. (Stage 7a)

Reproduces the Lorigan et al. (2025) salons-per-100k by IMD2025 quintile finding for the NE region specifically. This is the credibility anchor before the novel H2 analysis: if NE-level density-vs-IMD looks like the published England-wide finding, we know our salon enumeration is reasonable.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd

from schools_sunbeds import audit, config

config.ensure_dirs()
audit.verify_manifest()
TODAY = datetime.now(timezone.utc).strftime("%Y%m%d")
TODAY

In [ ]:
salons = gpd.read_file(sorted(config.DATA_PROCESSED.glob("salons_verified_union_*.gpkg"))[-1])
lsoa = gpd.read_file(sorted(config.DATA_PROCESSED.glob("lsoa_ne_*.gpkg"))[-1])
print(f"Verified salons: {len(salons)}")
print(f"NE LSOAs: {len(lsoa)}")

## 1. Salon density by IMD quintile (NE-wide)

In [ ]:
joined = gpd.sjoin(
    salons[["unique_id", "geometry"]],
    lsoa[["lsoa21cd", "imd_quintile", "pop_total", "pop_school_age", "geometry"]],
    how="left",
    predicate="within",
)
by_q = joined.groupby("imd_quintile")["unique_id"].count().rename("n_salons").to_frame()
pop_by_q = lsoa.groupby("imd_quintile")["pop_total"].sum().rename("pop_total")
child_by_q = lsoa.groupby("imd_quintile")["pop_school_age"].sum().rename("pop_school_age")

summary = by_q.join(pop_by_q).join(child_by_q)
summary["salons_per_100k_pop"] = (summary["n_salons"] * 1e5 / summary["pop_total"]).round(2)
summary["salons_per_100k_child"] = (summary["n_salons"] * 1e5 / summary["pop_school_age"]).round(2)
summary

In [ ]:
print("Q1 / Q5 ratio of salons per 100k total population:")
ratio = summary.loc[1, "salons_per_100k_pop"] / summary.loc[5, "salons_per_100k_pop"]
print(f"  {ratio:.2f}x")
print()
print("Q1 / Q5 ratio of salons per 100k school-age children:")
ratio_c = summary.loc[1, "salons_per_100k_child"] / summary.loc[5, "salons_per_100k_child"]
print(f"  {ratio_c:.2f}x")

## 2. Per-LA salon density

In [ ]:
by_la = joined.groupby("lsoa21cd").size()
lsoa_with_salons = lsoa.copy()
lsoa_with_salons["n_salons"] = lsoa_with_salons["lsoa21cd"].map(by_la).fillna(0).astype(int)
la = lsoa_with_salons.groupby("lad_code").agg(
    n_salons=("n_salons", "sum"),
    pop_total=("pop_total", "sum"),
    pop_school_age=("pop_school_age", "sum"),
)
la["salons_per_100k_pop"] = (la["n_salons"] * 1e5 / la["pop_total"]).round(2)
la["salons_per_100k_child"] = (la["n_salons"] * 1e5 / la["pop_school_age"]).round(2)
la.index = la.index.map(config.LA_NAMES_NE)
la.sort_values("salons_per_100k_pop", ascending=False)

In [ ]:
out_table = config.OUTPUTS_TABLES / f"T1_descriptive_replication_{TODAY}.csv"
summary.to_csv(out_table)
out_la_table = config.OUTPUTS_TABLES / f"T1b_per_la_density_{TODAY}.csv"
la.to_csv(out_la_table)
print("Wrote", out_table.relative_to(config.REPO_ROOT))
print("Wrote", out_la_table.relative_to(config.REPO_ROOT))